In [12]:
import geopandas as gpd
from pathlib import Path
from shapely import box

folder = Path("/Users/tenkanh2/data/Digiroad/UUSIMAA/UUSIMAA_2")
link_fp = folder / "DR_LINKKI_K.shp"
speeds_fp = folder / "DR_NOPEUSRAJOITUS_K.shp"


In [7]:
links = gpd.read_file(link_fp)
speeds = gpd.read_file(speeds_fp)

/Users/tenkanh2/micromamba/envs/python-gis-book/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(
/Users/tenkanh2/micromamba/envs/python-gis-book/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(


In [9]:
links.columns

Index(['LINK_ID', 'LINK_MMLID', 'SEGM_ID', 'KUNTAKOODI', 'HALLINN_LK',
       'TOIMINN_LK', 'LINKKITYYP', 'TIENUMERO', 'TIEOSANRO', 'SILTA_ALIK',
       'AJORATA', 'AET', 'LET', 'AJOSUUNTA', 'TIENIMI_SU', 'TIENIMI_RU',
       'TIENIM_PSA', 'TIENIM_KSA', 'TIENIM_ISA', 'ENS_TALO_O', 'ENS_TALO_V',
       'VIIM_TAL_O', 'VIIM_TAL_V', 'MUOKKAUSPV', 'SIJ_TARK', 'KOR_TARK',
       'ALKU_PAALU', 'LOPP_PAALU', 'GEOM_FLIP', 'LINK_TILA', 'GEOM_LAHDE',
       'MTK_TIE_LK', 'TIEN_KASVU', 'geometry'],
      dtype='object')

In [10]:
speeds.columns

Index(['ID', 'ALKU_M', 'LOPPU_M', 'VAIK_SUUNT', 'ARVO', 'MUOKKAUSPV',
       'LINK_ID', 'KUNTAKOODI', 'SEGM_ID', 'geometry'],
      dtype='object')

In [38]:
link_cols = ['LINK_ID', 'TOIMINN_LK', 'AJOSUUNTA', 'TIENIMI_SU', 'geometry']
speeds_cols = ["LINK_ID", "ARVO"]
data = links[link_cols].merge(speeds[speeds_cols], on="LINK_ID")

In [39]:
data = data.to_crs(epsg=4326)

In [40]:
data.head()

,LINK_ID,TOIMINN_LK,AJOSUUNTA,TIENIMI_SU,geometry,ARVO
0,fc68c955-9bb2-4808-beef-bbb236736fe4:1,6,2,None,"LINESTRING Z (25.10573 60.49079 59.558, 25.105...",40
1,fe6de144-f22b-4028-a390-796a9dbcb240:1,6,2,None,"LINESTRING Z (25.0233 60.23563 16.76, 25.02242...",20
2,114f2c12-39fe-42eb-956b-750915285291:1,5,2,None,"LINESTRING Z (25.09692 60.29349 49.7, 25.097 6...",40
3,362c6c08-fbd6-4864-bac9-5530d92e44b9:2,4,2,Arolan Kylätie,"LINESTRING Z (25.16814 60.53888 59.731, 25.168...",60
4,8e35e15a-9f21-4351-9e47-788afab3069a:1,6,2,Reiherinkuja,"LINESTRING Z (25.0389 60.1729 21.925, 25.03901...",40


In [21]:
bbox = gpd.GeoDataFrame(geometry=[box(24.896434,60.144048,24.983638,60.179578)], crs="EPSG:4326")
bbox.explore()

In [57]:
clipped = data.sjoin(bbox[["geometry"]])
clipped = clipped.reset_index(drop=True)
clipped["id"] = clipped.index
clipped = clipped.drop("index_right", axis=1)
clipped = clipped.drop("LINK_ID", axis=1)
clipped.head()

,TOIMINN_LK,AJOSUUNTA,TIENIMI_SU,geometry,ARVO,id
0,4,4,Uudenmaankatu,"LINESTRING Z (24.93434 60.16219 8.729, 24.9348...",30,0
1,4,3,Pitkäsilta,"LINESTRING Z (24.95013 60.1758 4.562, 24.95012...",40,1
2,5,3,Kruunuvuorenkatu,"LINESTRING Z (24.96903 60.16537 2.24, 24.96693...",30,2
3,5,2,Kristianinkatu,"LINESTRING Z (24.9567 60.1756 16.528, 24.95835...",30,3
4,5,2,Fabianinkatu,"LINESTRING Z (24.94979 60.16479 8.246, 24.9497...",30,4


In [58]:
# Drop roads that are not really used or are out of the area of interest
clipped = clipped[~clipped["id"].isin([1485, 1782, 559, 730, 2101, 1908, 1497, 19, 1014, 76, 1781, 321, 2428, 2347, 2079, 2344, 1472, 2447, 1780, 1332, 1798, 2566, 1779, 2130])]
clipped.explore()

In [60]:
clipped.columns

Index(['TOIMINN_LK', 'AJOSUUNTA', 'TIENIMI_SU', 'geometry', 'ARVO', 'id'], dtype='object')

In [61]:
# Translate
new_cols = {"LINK_ID": "id", "TOIMINN_LK": "road_class", "TIENIMI_SU": "name", "ARVO": "maxspeed", "AJOSUUNTA": "direction"}
clipped = clipped.rename(columns=new_cols)
clipped.head()

,road_class,direction,name,geometry,maxspeed,id
0,4,4,Uudenmaankatu,"LINESTRING Z (24.93434 60.16219 8.729, 24.9348...",30,0
1,4,3,Pitkäsilta,"LINESTRING Z (24.95013 60.1758 4.562, 24.95012...",40,1
2,5,3,Kruunuvuorenkatu,"LINESTRING Z (24.96903 60.16537 2.24, 24.96693...",30,2
3,5,2,Kristianinkatu,"LINESTRING Z (24.9567 60.1756 16.528, 24.95835...",30,3
4,5,2,Fabianinkatu,"LINESTRING Z (24.94979 60.16479 8.246, 24.9497...",30,4


In [62]:
clipped.columns

Index(['road_class', 'direction', 'name', 'geometry', 'maxspeed', 'id'], dtype='object')

In [63]:
col_order = ['id', 'direction', 'maxspeed', 'road_class', 'name', 'geometry']
clipped = clipped[col_order].to_crs(epsg=3067)

In [64]:
clipped.explore()

The speed-limit layer has several records per link (one per direction and per stretch where the limit changes), so the merge above duplicates link geometries. Keep one row per geometry, using the most common speed limit (lower value on ties).

In [ ]:
# Remove duplicated link geometries created by the many-to-one merge with the speed limits
wkb = clipped.geometry.to_wkb()
maxspeed = clipped.groupby(wkb)["maxspeed"].agg(lambda s: s.mode().iloc[0])
clipped = clipped[~wkb.duplicated()].copy()
clipped["maxspeed"] = wkb[~wkb.duplicated()].map(maxspeed).values
clipped = clipped.reset_index(drop=True)
clipped["id"] = clipped.index
print(len(clipped), "segments after removing duplicates")

In [65]:
clipped.to_file(folder/"digiroad_helsinki.gpkg")